# 008 — Qualification du dataset jardins
Profil configurable. Les modifications sont des hypothèses synthétiques à revoir, jamais des observations. Aucun appel API.

In [1]:
from pathlib import Path
import sys, json, copy, hashlib
import pandas as pd
cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "gdai-plant/config/dataset-scope-idf-v1.json").is_file())
PLANT = ROOT / "gdai-plant"
sys.path.insert(0, str(PLANT))
from matching.dataset_quality import audit
PROFILE = PLANT / "config/dataset-scope-idf-v1.json"
SOURCE = PLANT / "data/scenarios/garden-scenarios-climate-v1.json"
PLAN = SOURCE.with_name("garden-idf-review-plan-v1.json")
OUTPUT = SOURCE.with_name("garden-scenarios-idf-v1.json")
REPORT = SOURCE.with_name("garden-idf-qualification-report-v1.json")
profile = json.loads(PROFILE.read_text())
raw = SOURCE.read_bytes()
document = json.loads(raw)
report = audit(document, profile)
display(pd.DataFrame(report["issues"]))
print(report["flagged"], "/", report["total"])

,id,field,code,value
0,FR-PAR-01,region,MISSING,None
1,FR-PAR-01,region,OUT_OF_SCOPE,None
2,FR-PAR-01,minimum_winter_temperature_c,OUT_OF_RANGE,-8
3,FR-BRE-02,region,MISSING,None
4,FR-BRE-02,region,OUT_OF_SCOPE,None
...,...,...,...,...
117,FR-TEM-042,minimum_winter_temperature_c,OUT_OF_RANGE,-11
118,FR-TEM-043,region,OUT_OF_SCOPE,Grand Est
119,FR-TEM-043,climate_zone,OUT_OF_SCOPE,TEMPERATE_SEMI_CONTINENTAL
120,FR-TEM-043,minimum_winter_temperature_c,OUT_OF_RANGE,-13


48 / 50


## Revue explicite
Le plan contient tous les scénarios, y compris ceux sans anomalie automatique. Modifier les champs dans `changes`, documenter la raison et passer `reviewed` à true après revue géographique et horticole. Ne pas modifier les identifiants ni la provenance originale.

In [2]:
if not PLAN.exists():
    plan = [{"id": row["id"], "changes": {}, "reason": "", "reviewed": False}
            for row in document["scenarios"]]
    with PLAN.open("x", encoding="utf-8") as f:
        json.dump(plan, f, ensure_ascii=False, indent=2)
plan = json.loads(PLAN.read_text())
display(pd.DataFrame(plan))
print("Reviewed:", sum(p["reviewed"] is True for p in plan))

,id,changes,reason,reviewed
0,FR-PAR-01,{},,False
1,FR-BRE-02,{},,False
2,FR-AQU-03,{},,False
3,FR-CTR-04,{},,False
4,FR-EST-05,{},,False
5,FR-NOR-06,{},,False
6,FR-RHO-07,{},,False
7,FR-OCC-08,{},,False
8,FR-IDF-09,{},,False
9,FR-TEM-002,{},,False


Reviewed: 0


In [3]:
# Import des propositions sans validation automatique

PROPOSAL = SOURCE.with_name(
    "garden-idf-review-proposal-v1.json"
)

proposals = json.loads(PROPOSAL.read_text())
plan = json.loads(PLAN.read_text())

assert len(proposals) == 50
assert {p["id"] for p in proposals} == {
    p["id"] for p in plan
}

# Sauvegarde du plan précédent
BACKUP = PLAN.with_name(
    "garden-idf-review-plan-v1.backup.json"
)

if not BACKUP.exists():
    BACKUP.write_bytes(PLAN.read_bytes())

lookup = {p["id"]: p for p in proposals}

# Import uniquement pour les entrées non renseignées
for p in plan:
    if not p["reviewed"] and not p["changes"]:
        proposal = lookup[p["id"]]
        p["changes"] = proposal["changes"]
        p["reason"] = proposal["reason"]

PLAN.write_text(
    json.dumps(plan, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Nombre de jardins :", len(plan))
print("Propositions renseignées :",
      sum(bool(p["changes"]) for p in plan))
print("Jardins validés :",
      sum(p["reviewed"] is True for p in plan))

Nombre de jardins : 50
Propositions renseignées : 50
Jardins validés : 0


In [8]:
# Comparaison du dataset initial et des propositions

plan = json.loads(PLAN.read_text())

original = {
    row["id"]: row
    for row in document["scenarios"]
}

comparison = []

for p in plan:
    old = original[p["id"]]
    new = {**old, **p["changes"]}

    comparison.append({
        "id": p["id"],
        "ancienne_region": old.get("region"),
        "nouvelle_region": new.get("region"),
        "ancien_climat": old.get("climate_zone"),
        "nouveau_climat": new.get("climate_zone"),
        "ancienne_temperature":
            old.get("minimum_winter_temperature_c"),
        "nouvelle_temperature":
            new.get("minimum_winter_temperature_c"),
        "sol": new.get("soil_texture"),
        "humidite": new.get("soil_moisture_seasonality"),
        "exposition": new.get("sun_exposure"),
        "description": new.get("plausibility_reason"),
        "valide": p["reviewed"]
    })

df_review = pd.DataFrame(comparison)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

display(df_review)

,id,ancienne_region,nouvelle_region,ancien_climat,nouveau_climat,ancienne_temperature,nouvelle_temperature,sol,humidite,exposition,description,valide
0,FR-PAR-01,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-8,-6,LOAM,WELL_DRAINED,FULL_SUN,"Synthetic Île-de-France scenario: Suburban front lawn border. Existing soil, exposure and planting constraints retained.",False
1,FR-BRE-02,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-8,-6,MIXED,SEASONAL_WET,PARTIAL_SHADE,"Synthetic Île-de-France scenario: Fence-sheltered residential border. Existing soil, exposure and planting constraints retained.",False
2,FR-AQU-03,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-8,-5,MIXED,DRY_SUMMER,FULL_SUN,"Synthetic Île-de-France scenario: Sunny paved patio border. Existing soil, exposure and planting constraints retained.",False
3,FR-CTR-04,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-7,-7,MIXED,WELL_DRAINED,FULL_SHADE,"Synthetic Île-de-France scenario: North-facing house foundation. Existing soil, exposure and planting constraints retained.",False
4,FR-EST-05,None,Île-de-France,TEMPERATE_SEMI_CONTINENTAL,TEMPERATE_TRANSITIONAL,-12,-3,SILT,SEASONAL_WET,FULL_SUN,"Synthetic Île-de-France scenario: Open alluvial garden perimeter. Existing soil, exposure and planting constraints retained.",False
5,FR-NOR-06,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-9,-7,MIXED,WELL_DRAINED,PARTIAL_SHADE,"Synthetic Île-de-France scenario: Gently sloping side yard. Existing soil, exposure and planting constraints retained.",False
6,FR-RHO-07,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-8,-6,CLAY,SEASONAL_WET,FULL_SHADE,"Synthetic Île-de-France scenario: Enclosed shaded clay courtyard. Existing soil, exposure and planting constraints retained.",False
7,FR-OCC-08,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-6,-6,STONY,DRY_SUMMER,FULL_SUN,"Synthetic Île-de-France scenario: Gravel driveway border. Existing soil, exposure and planting constraints retained.",False
8,FR-IDF-09,None,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-9,-4,CLAY,SEASONAL_WET,FULL_SUN,"Synthetic Île-de-France scenario: Roof-runoff swale. Existing soil, exposure and planting constraints retained.",False
9,FR-TEM-002,Bretagne,Île-de-France,TEMPERATE_TRANSITIONAL,TEMPERATE_TRANSITIONAL,-8,-3,SAND,WELL_DRAINED,PARTIAL_SHADE,"Synthetic Île-de-France scenario: Sheltered fence border. Existing soil, exposure and planting constraints retained.",False


In [6]:
plan = json.loads(PLAN.read_text())

for p in plan:
    if p["id"] == "FR-PAR-01":
        p["changes"]["minimum_winter_temperature_c"] = -6
        p["reason"] += " Température ajustée après revue."
        p["reviewed"] = False

PLAN.write_text(
    json.dumps(plan, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

29044

In [7]:
plan = json.loads(PLAN.read_text())
ids = [r["id"] for r in document["scenarios"]]
assert len(plan) == len(ids) == len(set(ids))
assert {p["id"] for p in plan} == set(ids)
assert all(p["reviewed"] is True and p["reason"].strip() for p in plan)
updated = copy.deepcopy(document)
lookup = {p["id"]: p for p in plan}
allowed = set(profile["allowed_values"]) | set(profile["numeric_ranges"]) | {"region", "plausibility_reason"}
for row in updated["scenarios"]:
    p = lookup[row["id"]]
    assert set(p["changes"]) <= allowed
    row.update(p["changes"])
final = audit(updated, profile)
assert final["flagged"] == 0 and final["target_count_met"], final
assert SOURCE.read_bytes() == raw
updated["qualification"] = {"profile_id": profile["profile_id"], "status": "SYNTHETIC_REVIEWED", "source_sha256": hashlib.sha256(raw).hexdigest(), "profile_sha256": hashlib.sha256(PROFILE.read_bytes()).hexdigest()}
assert not OUTPUT.exists() and not REPORT.exists()
encoded = json.dumps(updated, ensure_ascii=False, indent=2)
final["output_sha256"] = hashlib.sha256(encoded.encode()).hexdigest()
final["review_plan"] = plan
OUTPUT.write_text(encoded, encoding="utf-8")
REPORT.write_text(json.dumps(final, ensure_ascii=False, indent=2), encoding="utf-8")
print(OUTPUT, REPORT)

AssertionError: 